# Ontology: concepts, queries, and overrides

The ontology is the workspace's semantic layer: concepts are the shared
meaning of columns across sources, and the query engine joins, filters, and
aggregates through them so you never hand-write reconciliation SQL. This
notebook uploads a small dataset, works with the concepts the ingest built,
queries through them, and overrides what the profiler detected.

In [1]:
import os

import numpy as np
import pandas as pd

import rootcause as rc

rc.login(base_url=os.environ.get("ROOTCAUSE_BASE_URL", "https://platform.rootcause.ai"))
ws = rc.workspace("ontology-tour", create=True)

## Concepts arrive with the data

Every upload gets concepts during ingest — profiled, typed, and named:

In [2]:
rng = np.random.default_rng(29)
n = 240
marketing = rng.normal(50, 12, n)
leads = 2.1 * marketing + rng.normal(0, 8, n)
frame = pd.DataFrame({
    "region": rng.choice(["emea", "apac", "amer"], n),
    "marketing_spend": marketing.round(2),
    "leads": leads.round(1),
    "revenue": (3.4 * leads + rng.normal(0, 15, n)).round(1),
    # a running total with occasional dedup corrections — it dips, so the
    # profiler will rightly refuse to call it monotonic on the data alone
    "cumulative_signups": np.cumsum(rng.integers(1, 9, n)) - np.where(rng.random(n) < 0.06, 4, 0),
})
source = ws.upload(frame, "campaign-weeks")

# ontology analysis runs just after ingest; wait for the concepts to land
import time
onto = ws.ontology
while len(onto.concepts) < frame.shape[1]:
    time.sleep(2)
onto.concepts

,id,name,type,classification,sources
0,1DU2kACKzwmwjKNmOAoac,Region,Category,location,1
1,1YtV9Wod6SiwoyDaMj02Y,Revenue,Number,NaN,1
2,Hj8inuLkU9RVzTnUaBhIo,Cumulative Signups,Number,NaN,1
3,W7dNhHpqrvmB9RoIf40Hk,Marketing Spend,Number,NaN,1
4,oSrjfpd9M6yHzmXDnDkU0,Leads,Number,NaN,1


`onto[...]` hands back a **concept handle** — resolve once, then every
operation lives on the object. If a name matches more than one concept, the
lookup refuses with the candidates listed (`onto.matching(name)` disambiguates).

In [3]:
signups = onto["Cumulative Signups"]
signups

Concept('Cumulative Signups', type=Number)

## Query through concepts

`sql()` runs Anchor SQL — SQL over concepts, not tables. Concepts go by
quoted name and the ontology compiles the statement against the underlying
sources, so there is nothing to FROM and no joins to write:

In [4]:
onto.sql(
    'SELECT "Region", "Marketing Spend", "Revenue" WHERE "Revenue" >= 400 ORDER BY "Revenue"'
).to_frame().tail(5)

,Region,Marketing Spend,Revenue
75,apac,76.50,557.2
76,apac,70.05,559.6
77,apac,82.22,586.8
78,apac,82.95,614.9
79,emea,85.79,632.5


Aggregates read as SQL — GROUP BY, HAVING, ORDER BY and LIMIT all work,
and metrics defined in the workspace are selectable by name the same way:

In [5]:
onto.sql('SELECT "Region", avg("Revenue"), avg("Marketing Spend") GROUP BY "Region" ORDER BY "Region"').to_frame()

,Region,avg_Revenue,avg_Marketing Spend
0,amer,352.153165,49.554684
1,apac,365.862069,51.570460
2,emea,359.852703,50.586081


The engine also answers what there is to query: `SHOW CONCEPTS` (with an
optional `LIKE '%pattern%'`), `SHOW METRICS`, `SHOW SOURCES`, and `DESCRIBE`:

In [6]:
onto.sql('DESCRIBE "Cumulative Signups"').to_frame()

,property,value
0,name,Cumulative Signups
1,kind,standard
2,classification,
3,type,Number
4,unit,
5,source,campaign-weeks (1).cumulative_signups
6,anchors,Region


A statement the engine refuses raises `AnchorSqlError` — a structured
compile error carrying the offending span, ranked near-miss `candidates`,
and, for syntax slips, a corrected statement in `suggested_query`:

In [7]:
from rootcause import AnchorSqlError

try:
    onto.sql('SELECT avg("Revenu")')
except AnchorSqlError as err:
    caught = err
    print(err)
    print("candidates:", [(c["name"], c["score"]) for c in err.candidates])
    print("suggested:", err.suggested_query)

[unknown_concept] Unknown concept "Revenu". Closest matches: Revenue
candidates: [('Revenue', 0.923)]
suggested: None


The repair loop is mechanical — feed the top candidate straight back:

In [8]:
fixed = caught.candidates[0]["name"]
onto.sql(f'SELECT avg("{fixed}")').to_frame()

,avg_Revenue
0,359.496667


## Override what the profiler detected

The running total dips whenever duplicates are removed, so on the data alone
the profiler rightly refuses to flag it monotonic. Semantically it can never
go down — and the model should know that. Override it on the handle: fields
are set and **locked**, so future ingests preserve your values while the
detected ones keep shadowing underneath:

In [9]:
signups.override(monotonically_increasing=True, description="Running total of signups")
signups.locks

,field,value,detected
0,isMonotonicallyIncreasing,True,False


Forecasts apply a running clamp to any node whose concept is flagged
monotonically increasing — a cumulative metric can never forecast downward.
Other keywords cover units, value ranges, fill strategies, categories, and the
concept-level role (`suggested_role="target"` sets the default for every
future twin built over the concept). Overriding a field to the value the
profiler already detected is a deliberate no-op — locks exist only where you
actually disagree.

`revert()` hands fields back to the profiler — detected values restored,
locks lifted:

In [10]:
signups.revert()
signups.locks

,field,value,detected


From here the concepts feed everything else: twins built over this workspace
inherit the roles and metadata, and [temporal-panel.ipynb](temporal-panel.ipynb)
shows the modelling side.